#Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import trim,col

In [0]:
rename_map = {
    'cst_id': 'customer_id',
    'cst_key': 'customer_number',
    'cst_firstname': 'first_name',
    'cst_lastname': 'last_name',
    'cst_marital_status': 'marital_status',
    'cst_gndr': 'gender',
    'cst_create_date': 'create_date'
}



#Reading Data From Bronze Layer

In [0]:
df = spark.table('workspace.bronze.crm_cust_info')

#Data Cleaning

##Removing missing Values and duplication

In [0]:
query = '''
SELECT 
    cst_id,
    cst_key,
    cst_firstname,
    cst_lastname,
    cst_marital_status,
    cst_gndr,
    cst_create_date
    From (
        SELECT 
            *,
            ROW_NUMBER() OVER(PARTITION BY cst_id ORDER BY cst_create_date DESC) AS is_unique
            FROM bronze.crm_cust_info) t
    WHERE is_unique = 1 AND cst_id IS NOT NULL
'''

df = spark.sql(query)

##Trimming String columns

In [0]:
for column in df.dtypes:
    if column[1] == 'string':
        df = df.withColumn(column[0],trim(col(column[0])))

##Standardization

###Value Mapping (e.g., S → Single)


In [0]:
df = (
df.withColumn(
    'cst_marital_status',
    F.when(F.upper(F.trim('cst_marital_status')) == 'S', 'Single')
    .when(F.upper(F.trim('cst_marital_status')) == 'M', 'Married')
    .otherwise('Unknown')
    )
.withColumn(
    'cst_gndr',
    F.when(F.upper(F.trim('cst_gndr')) == 'M', 'Male')
    .when(F.upper(F.trim('cst_gndr')) == 'F', 'Female')
    .otherwise('Unknown')
    )
)


###Column Naming Standardization

In [0]:
for old_name,new_name in rename_map.items():
    df = df.withColumnRenamed(old_name,new_name)


#Loading Data Into Silver Layer

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('silver.crm_customer_info')